# Build `earthmover-public/virtual-omics`

Rebuild three published CZ CELLxGENE studies as **virtual references** — copying no
expression data — into one Arraylake-backed Icechunk repo, one group per study:

| group | study | cells |
|---|---|---|
| `allen/immune_health_atlas` | Human Immune Health Atlas | 1.82M |
| `allen/immunobiology_of_aging` | Immunobiology of Aging Cohort | 3.76M |
| `ocular_surface_atlas` | Ocular Surface Atlas | 815k |

The repo stores each file's HDF5 byte ranges plus the small native metadata
(`obs`/`var`/`indptr`); slicing an opened AnnData later streams only the cells you
touch, straight from the CELLxGENE HTTPS host.

**Transient-container design** — this is the fresh repo that replaces the bricked
`virtual-omics-demo`. `virtualize_matrix(repo=...)` installs the source's
`https` virtual-chunk container *in memory* and never calls `save_config`, so the
persisted repo config stays clean and Arraylake's `/open` won't 403 it. Readers
re-supply the container transiently from the committed `source_files` attr, so
`open_anndata(client.get_repo(REPO_NAME), group=...)` just works.

## Prerequisites

- **Authenticated Arraylake session** — `arraylake auth login`, or a token in the
  environment. `Client()` picks it up.
- **Indexing downloads** — building walks each file's HDF5 chunk index, which needs
  the file locally. Each source is fetched once (parallel range GETs) into
  `cache_dir` and **reused on re-run**, not deleted. Sizes on disk: immune ~20 GB,
  aging ~40 GB, ocular ~4 GB. The immune + ocular copies are already cached under
  `data/`; the aging file needs a fresh fetch.
- **RAM** — `obs` is concatenated and written natively, so the node needs enough
  memory to hold the largest frame (~3.76M rows for the aging cohort). Intended to
  run on Coiled or a roomy box; a laptop is fine for immune + ocular alone.

No expression values are read during the build — only the chunk index and the small
metadata arrays.

In [1]:
import time
from pathlib import Path

from arraylake import Client

from virtual_h5ad import open_anndata, virtualize_matrix
from virtual_h5ad._open import open_root
from virtual_h5ad._sources import MATRIX_GROUP

CELLXGENE = "https://datasets.cellxgene.cziscience.com/"
REPO_NAME = "earthmover-public/virtual-omics"  # the fresh repo (NOT the bricked -demo)

# (group, single full-atlas CELLxGENE file, label). Each is one range-servable file;
# the per-cell-type partitions are just the same cells split, so we take the whole.
DATASETS = [
    (
        "allen/immune_health_atlas",
        CELLXGENE + "c1ce03ce-4b71-4b8d-b695-c3cec99cdfc2.h5ad",
        "Human Immune Health Atlas (1.82M cells)",
    ),
    (
        "allen/immunobiology_of_aging",
        CELLXGENE + "924554e2-d893-4015-be86-017e96fab55b.h5ad",
        "Immunobiology of Aging (3.76M cells)",
    ),
    (
        "ocular_surface_atlas",
        CELLXGENE + "30127f02-d0b9-4b56-bf4c-e5b77e4989bd.h5ad",
        "Ocular surface atlas (815k cells)",
    ),
]

# Indexing downloads land here and are NOT deleted, so a re-run reuses them. A local
# ocular copy under data/ocular_cache is reused if present.
CACHE = Path("data/_source_cache")
OCULAR_CACHE = Path("data/ocular_cache")
CACHE.mkdir(parents=True, exist_ok=True)

In [2]:
# Arraylake hands back an icechunk.Repository. We do NOT pass
# authorize_virtual_chunk_access (that maps prefixes to Arraylake S3 bucket nicknames,
# for private sources): CELLxGENE is public HTTPS, and building only writes refs +
# reads native metadata. virtualize_matrix installs the HTTP container transiently.
client = Client()
repo = client.get_or_create_repo(REPO_NAME)
repo

<icechunk.Repository (v2)>
storage:
    <icechunk.Storage>
    type: S3 (native)
    bucket: ems-earthmover-public-0f0978f1
    prefix: 4anu5ehz_virtual_omics
    region: us-east-1
config: <RepositoryConfig ...>

## Build

One `virtualize_matrix` call per study. Each references `X` (`data`/`indices`) and any
shared dense `obsm` as rectilinear virtual arrays, writes `indptr` + `obs`/`var`
natively, and commits — with the `https` container installed transiently (never
persisted). `write_concat_virtual_array` raises if any ref is rejected, so a
mis-installed container fails the build **loudly** instead of committing all-zeros.

`BUILD_ONLY_CACHED = True` (default) builds only the groups whose source file is
already downloaded under `data/` — right now that's **immune + ocular** (both cached
byte-complete); **aging** is skipped with a note until its ~40 GB file is fetched. Set
it to `False` to download and index whatever's missing. Re-runs reuse the cache and
rewrite each group cleanly (`overwrite`-safe).

In [3]:
BUILD_ONLY_CACHED = True  # skip groups whose source isn't downloaded (e.g. aging, ~40 GB)

built = []
for i, (group, url, label) in enumerate(DATASETS, 1):
    cdir = (
        OCULAR_CACHE
        if group == "ocular_surface_atlas" and OCULAR_CACHE.is_dir()
        else CACHE
    )
    cached = (cdir / Path(url).name).exists()
    if BUILD_ONLY_CACHED and not cached:
        print(
            f"[{i}/{len(DATASETS)}] skipping {label} — source not cached "
            f"({Path(url).name}); set BUILD_ONLY_CACHED = False to fetch & index it.",
            flush=True,
        )
        continue
    print(
        f"[{i}/{len(DATASETS)}] virtualizing {label} → {REPO_NAME} /{group} "
        + ("(cached)" if cached else "(downloading to index)") + "…",
        flush=True,
    )
    t = time.time()
    snapshot = virtualize_matrix(url, group=group, repo=repo, cache_dir=str(cdir))
    built.append(group)
    print(f"        ✓ {group} committed in {time.time() - t:.0f}s ({snapshot})", flush=True)

print(f"\nbuilt groups: {built or '(none)'}")

[1/3] virtualizing Human Immune Health Atlas (1.82M cells) → earthmover-public/virtual-omics /allen/immune_health_atlas (cached)…
        ✓ allen/immune_health_atlas committed in 73s (8R9AKHNP3DNG8B731930)
[2/3] skipping Immunobiology of Aging (3.76M cells) — source not cached (924554e2-d893-4015-be86-017e96fab55b.h5ad); set BUILD_ONLY_CACHED = False to fetch & index it.
[3/3] virtualizing Ocular surface atlas (815k cells) → earthmover-public/virtual-omics /ocular_surface_atlas (cached)…
        ✓ ocular_surface_atlas committed in 30s (XWDA6PK5FBDDDSK3GCJ0)

built groups: ['allen/immune_health_atlas', 'ocular_surface_atlas']


## Verify — non-zero `X` read-back

The check that would have caught the brick. Shapes alone are **not** enough — the
bricked repo had perfect shapes while `X` read back all-zeros (its refs had been
dropped for want of a container). So here we re-fetch the repo through the client
(proving `/open` works and the transient read container resolves), then stream a
small row slice of each group's `X` and assert it is genuinely non-zero.

In [12]:
import anndata.io as aio
import numpy as np

N_CHECK = 500  # cells to stream per group for the non-zero assertion

repo_ro = client.get_repo(REPO_NAME)  # fresh handle: goes through /open, like a reader
checked = 0
for group, _url, label in DATASETS:
    try:
        root = open_root(repo_ro, group)
    except KeyError:
        print(f"  {group:34} (not in repo — skipped)", flush=True)
        continue
    shape = tuple(int(x) for x in root[MATRIX_GROUP].attrs["shape"])
    obsm = list(root["obsm"].array_keys()) if "obsm" in root else []
    x = aio.sparse_dataset(root[MATRIX_GROUP])
    sub = x[:N_CHECK]  # streams only these rows' referenced bytes
    # The brick symptom is real *structure* (nnz, from native indptr/indices) but all
    # stored VALUES zero — dropped refs read back as fill_value. So assert on the
    # magnitude of the stored data, not its signed sum: some layers are mean-centered
    # (negative values), so a genuine slice can sum to <= 0.
    mag = float(np.abs(sub.data).sum())
    assert sub.nnz > 0 and mag > 0, f"{group}: X read back all-zeros (dropped refs)!"
    checked += 1
    print(
        f"  {group:34} shape={shape}  obsm={obsm}\n"
        f"       first {N_CHECK} rows: nnz={sub.nnz:,}  |data|sum={mag:,.0f}  "
        f"range=[{sub.data.min():.3f}, {sub.data.max():.3f}]  ✓",
        flush=True,
    )
print(f"\nverified {checked} group(s) → {REPO_NAME}: X reads back real (non-zero) data")

  allen/immune_health_atlas          shape=(1821725, 32357)  obsm=['X_pca_harmony', 'X_pca', 'X_umap']
       first 500 rows: nnz=612,500  |data|sum=283,083  range=[-3.119, 37.351]  ✓


  allen/immunobiology_of_aging       (not in repo — skipped)


  ocular_surface_atlas               shape=(815769, 35477)  obsm=['X_scVI', 'X_umap']
       first 500 rows: nnz=2,037,306  |data|sum=1,326,871  range=[0.067, 6.908]  ✓



verified 2 group(s) → earthmover-public/virtual-omics: X reads back real (non-zero) data


## Use it

How a consumer opens a group — ordinary lazy AnnData, backed by the repo. Opening
reads no expression data; slicing streams only the touched cells from CELLxGENE.

In [10]:
%%time
adata = open_anndata(client.get_repo(REPO_NAME), group="ocular_surface_atlas")
print(adata)

# Materialize a subset — only these cells' chunks are fetched.
sub = adata[:1000].to_memory()
print(sub.shape, "nnz:", sub.X.nnz, "sum:", float(sub.X.sum()))

AnnData object with n_obs × n_vars = 815769 × 35477
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_dissociation_time', 'suspension_uuid', 'suspension_type', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'pANN', 'study', 'sampleid', 'published_annotation', 'majorclass', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'o

In [11]:
sub

AnnData object with n_obs × n_vars = 1000 × 35477
    obs: 'reference_genome', 'gene_annotation_version', 'alignment_software', 'intronic_reads_counted', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_living_at_sample_collection', 'sample_id', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_collection_method', 'tissue_source', 'tissue_type', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_dissociation_time', 'suspension_uuid', 'suspension_type', 'library_id', 'assay_ontology_term_id', 'sequenced_fragment', 'institute', 'library_id_repository', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'pANN', 'study', 'sampleid', 'published_annotation', 'majorclass', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'obs